# 02 -- Windows and export

Extract a window, compute several things, export to a self-describing .h5.

In [1]:
%matplotlib inline

import numpy as np
from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

# The example data folder (edit to your path if needed).
DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"

# A short window keeps every example fast (one ~10 min file).
WSTART, WLEN = "2026-05-31 15:00:00", "10min"

In [2]:
ds = SensorDataset(DATA, verbose=True)
ds.devices

------------------------------------------------------------
SensorDataset
------------------------------------------------------------
path        : C:\Users\ppala\Desktop\02_31MAY2026
files       : 32
time span   : 2026-05-31 14:52:12  ->  2026-05-31 20:02:13
duration    : 18601.0 s
devices     : MNAT0031, MNAT0034, MOF00134, MOF00135, MOF00136
fs / dt     : 252.5885 Hz / 0.003959 s
------------------------------------------------------------
axes (per sensor):
  MNAT0031   -> (3, 1, 5)
  MNAT0034   -> (3, 1, 5)
  MOF00134   -> (0, 1, 2)
  MOF00135   -> (3, 1, 5)
  MOF00136   -> (3, 1, 5)
------------------------------------------------------------
on-disk size: 782.17 MB
RAM         : used 33.35 GB / avail 0.16 GB (100%)
------------------------------------------------------------


['MNAT0031', 'MNAT0034', 'MOF00134', 'MOF00135', 'MOF00136']

## Window by start + length

In [3]:
w = ds.MOF00135.window(WSTART, WLEN)
sig = w.signal(components="x")
print("window n =", sig.n, " dt =", round(sig.dt, 6))

[signal] MOF00135 n=149240 dt=0.004021 comps=x
window n = 149240  dt = 0.004021


C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\core\h5_reader.py:45: UserWarning: H5Reader: axes (3, 1, 5) for 'MOF00135' exceed the 3 available columns; falling back to (0, 1, 2)
  warnings.warn(


## Process and compute

In [4]:
wf = w.filter(0.1, 24.9).derive()
wf.newmark(component="all")
wf.arias(component="all")
wf.fourier(component="x")
wf.psd(component="x")
ds.cache_summary()

C:\Dropbox\01. Brain\11. GitHub\ladruno_ASDEA_sensors\src\asdea_sensors\derive\filters.py:49: UserWarning: obspy not available; falling back to the scipy engine
  warnings.warn("obspy not available; falling back to the scipy engine")


[newmark] MOF00135 comp=x zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)


[newmark] MOF00135 comp=y zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)


[newmark] MOF00135 comp=z zeta=0.05 Tmax=5.01 dT=0.01 -> 501 periods (computed)
[arias] MOF00135 comp=x low=5 high=95 (computed)
[arias] MOF00135 comp=y low=5 high=95 (computed)
[arias] MOF00135 comp=z low=5 high=95 (computed)
[fourier] MOF00135 comp=x nfreq=4 smooth=None (computed)
[psd] MOF00135 comp=x nperseg=256 (computed)
------------------------------------------------------------
cache: 8 entries, 18.29 MB
hint: call clear_cache() to free it
------------------------------------------------------------


## Export the window with everything

In [5]:
import tempfile, os
out = os.path.join(tempfile.gettempdir(), "MOF00135_window.h5")
wf.export_h5(out)
print("written:", out)

written: C:\Users\ppala\AppData\Local\Temp\claude\MOF00135_window.h5


## Read it back

In [6]:
from asdea_sensors.io.results_file import ResultsFile
r = ResultsFile(out)
print("provenance units:", r.provenance.get("units"), " | pipeline:", r.provenance.get("pipeline"))
print("devices:", r.devices)
print("analyses for MOF00135:", r.analyses("MOF00135"))

provenance units: SI  | pipeline: 
devices: ['MOF00135']
analyses for MOF00135: ['arias', 'arias_2', 'arias_3', 'fourier', 'newmark', 'newmark_2', 'newmark_3', 'psd']
